In [2]:
import pandas as pd
import pyarrow.dataset as pads
import os
import numpy as np

In [ ]:
#Function provided by SMT to read in the data from the .csv format into large meta tables
def readDataSubset(table_type, data_path=''):
    '''
    Takes a the set of tables from Data set and
    transforms into panda data frame for manipulation
    
    Input:
    -     table_type   = Set of tables of interest from Data set [str]
    -     data_path    = File Path which data is located in Local device, change as necessary[str]
    
    '''
    if table_type not in ['ball-positions', 'ball-events', 'player-positions', 'lineups', 'game-info']:
        print("Invalid data subset name. Please try again with a valid data subset.")
        return -1

    if table_type == 'lineups' or table_type == 'game-info':
        return pads.dataset(source = os.path.join(os.path.dirname(__name__), data_path, table_type + '.csv'), format = 'csv')
    else:
        return pads.dataset(source = os.path.join(os.path.dirname(__name__), data_path, table_type), format = 'csv', partitioning = ['home_team', 'away_team', 'year', 'day'])

In [4]:
#Reading in the lineups csv to obtain info on runners on base
lineups = readDataSubset('lineups')

#Creating filtering criteria to get plays with a runner on 3rd
filter_criteria_1 = (pads.field('on_3b') != 'NA')


#Loading the lineups information with runners on 3rd into a pandas dataframe
filtered_df = lineups.to_table(filter=filter_criteria_1).to_pandas()

#Creation of play_str for the lineups based df
#play_string allows us to filter by the game_string and play_per_game effectively
filtered_df['play_per_game'] = filtered_df['play_per_game'].astype(float)
filtered_df['play_per_game'] = filtered_df['play_per_game'].astype(str)
filtered_df['play_string'] = filtered_df['game_string'] + '_' + filtered_df['play_per_game']

#Filtering down the plays with a runner on 3rd base to only include play_string and the id of the runner on 3rd base
interested = filtered_df[['play_string', 'on_3b']]



In [5]:

#Read in ball_events to a dataframe
ball_events_subset = readDataSubset('ball-events')
ball_events = ball_events_subset.to_table().to_pandas()

#Creation of play_str to ensure querying returns the proper result for game_string and play_per_game
ball_events['play_per_game'] = ball_events['play_per_game'].astype(float)
ball_events['play_per_game'] = ball_events['play_per_game'].astype(str)
ball_events['play_string'] = ball_events['game_string'] + '_' + ball_events['play_per_game']

#Filtering the ball_events to only include information on the plays where there is a runner on third base
interested_ball_events = ball_events[
     ball_events['play_string'].isin(interested['play_string'])
      ]


In [6]:
#Sorting the timestamps to ensure that all timestamps are in proper sequential order
interested_ball_events = interested_ball_events.sort_values(by=['play_string', 'timestamp'], ascending=[True, True])

In [7]:
#Taking the list of plays with a runner on third and filtering down for the play that where the ball is hit and then subsequently acquired
#without hitting the ground and the play does not end after the ball is acquired.

#Isolate plays where outfielders acquire the ball after it's hit that do not end the inning
outfield_catch = interested_ball_events[
    (interested_ball_events['player_id'].isin([7,8,9])) &
    (interested_ball_events['ball_eventcode'] == 2) &
    (interested_ball_events['ball_eventcode'].shift(1) == 4) &
    (interested_ball_events['ball_eventcode'].shift(-1) != 5)
    ]


In [8]:
#Keeping just the play information and timestamp of the catch
outfield_catch = outfield_catch[['play_string', 'timestamp']]

#Renaming the timestamp of when the ball is caught to catch_timestamp
outfield_catch = outfield_catch.rename(columns={'timestamp': 'catch_timestamp'})

In [9]:
#Read in player_positions
player_positions_subset = readDataSubset('player-positions')

#Create a filter to only include the player position data for runners on 3rd base and load it into a dataframe
runner_on_3rd = (pads.field('player_id') == 13)
player_positions = player_positions_subset.to_table(filter=runner_on_3rd).to_pandas()

#Creation of play_str to ensure querying returns the proper result for game string and play id
player_positions['play_per_game'] = player_positions['play_per_game'].astype(float)
player_positions['play_per_game'] = player_positions['play_per_game'].astype(str)
player_positions['play_string'] = player_positions['game_string'] + '_' + player_positions['play_per_game']
player_positions.head()

#Filtering the player_positions down to just the plays that we had previously identified as plays of interest
third_base_runner= player_positions[
     player_positions['play_string'].isin(outfield_catch['play_string'])
      ]


In [ ]:
#Converting the cartesian plane to polar plane for plays with runners on third for easier measurement
third_base_runner['field_r']=np.sqrt(third_base_runner['field_x']**2 + third_base_runner['field_y']**2)
third_base_runner['field_theta']=np.arctan2(third_base_runner['field_y'],third_base_runner['field_x'])*(180/(np.pi))

In [11]:
#Joining the third base runner dataframe with the dataframe of all the catches on the same play_string
#This gives us one dataframe of player positions of runners on 3rd base when a ball is caught by the outfielder with the catch_timestamp
runner = pd.merge(third_base_runner, outfield_catch, on='play_string', how='left')

#Filtering down the previous dataframe to only include the player_positions where the timestamp is greater than or equal to the catch_timestamp
#In order to calculate Go Time we only need the information for after the catch was made as a tag back to the base must occur after the catch
runner = runner[
    runner['timestamp'] >= runner['catch_timestamp']
]

#Select only the columns that will be used in future analysis 
runner = runner[['play_string', 'timestamp', 'field_x', 'field_y', 'field_r', 'field_theta', 'catch_timestamp']]

In [12]:
#Create a boolean (True or False) list that tests whether each play has a competitve run to home
#A competitve run home is where the player crosses the 60 foot mark towards home
comp_run_TF = runner.groupby('play_string').min()['field_r'] < 60

#Move the boolean list into a dataframe that only keeps the play_strings where the runner makes a competitive run home
comp_run_string = comp_run_TF.to_frame()
comp_run_string = comp_run_string[comp_run_string['field_r'] == True].reset_index()

#Create a new dataframe to only include the player_position and catch_timestamp for plays that had a competitive run
comp_run = runner[
    runner['play_string'].isin(comp_run_string['play_string'])
]


In [13]:
#Create a dataframe that takes the columns needed to compute the constant velocity to complete the time the runner is off the base
time_off_base = comp_run[['play_string', 'timestamp', 'field_x', 'field_y']]


In [14]:
#Find the origin of each player when the ball is caught under the assumption that the player is on the base at the time of the ball being caught

#Take the earliest time stamp for each play
origin = time_off_base.groupby('play_string').min()['timestamp']

#Take those timestamps and place them in a dataframe
origin = origin.to_frame().reset_index()
origin.columns = ['play_string', 'timestamp']

#Join back with the information on the field_x and field_y of that player and rename it as the origin location of the player
origin = origin.merge(time_off_base, how='left', on=['play_string', 'timestamp'])
origin = origin.drop(columns='timestamp')
origin.columns = ['play_string', 'origin_x', 'origin_y']

In [ ]:
#Join the origin dataframe to the instant_velo dataframe to be able to consistently reference the same origin point for each unique play
time_off_base = time_off_base.merge(origin, how='left', on='play_string')


In [16]:
#Calculate the players change in x and y relative to the origin position
time_off_base['change_x'] = time_off_base['field_x'] - time_off_base['origin_x']
time_off_base['change_y'] = time_off_base['field_y'] - time_off_base['origin_y']

#Use the distance formula to calculate the displacement of the runner at each point
time_off_base['displacement'] = np.sqrt(time_off_base['change_x']**2 + time_off_base['change_y']**2)

#Creation of other variables necessary for calculation to determine the timestamp the runner is off the base
time_off_base['change_dis'] = time_off_base.groupby('play_string').diff()['displacement']
time_off_base['dt'] = time_off_base.groupby('play_string').diff()['timestamp']

#Calculate the additional time (ms) it would take the runner to get to 2 feet of displacement assuming a constant velocity 
#between the prior and current data point
time_off_base['time_to_2ft'] = ((2 - time_off_base['displacement'].shift(1))/time_off_base['change_dis']) * time_off_base['dt']

#Calculating the timestamp that player reached 2 feet from their origin location using the above calculation plus the timestamp of 
#the data point immediately prior to the player reaching 2 feet
time_off_base['off_base'] = time_off_base['timestamp'].shift(1) + (time_off_base['time_to_2ft'])


In [ ]:
#Filter the above dataframe to only include the data points where the displacement is greater than or equal to 2 feet 
#and the displacement is less than or equal to 3.5 feet. This is needed because the gaps in the data caused by the 20 fps frame rate mean that the exact timestamp
#a player crosses the 2 foot mark is unknown. Instead, we have to use the timestamp closest to 2 feet and use constant velocity to determine the GO TIME. 
#For more information see Appendix A1 in our paper.
time_off_base = time_off_base[
    (time_off_base['displacement'] <= 3.5) &
    (time_off_base['displacement'] >= 2.0)
]

In [ ]:
#Taking a list of each plays closest minimum displacement which is the first displacement greater than 2 feet for each individual play
minimum_displacement = time_off_base.groupby('play_string').min()['displacement']

#Create a dataframe of these displacements 
minimum_displacement = minimum_displacement.to_frame().reset_index()


In [ ]:
#Create a dataframe with the timestamp that the player has left the base and is considered to have gone home
off_base = time_off_base.merge(minimum_displacement, how="inner", on=["play_string","displacement"])

#Keep only the time the runner is off the base and the play string
off_base = off_base[['play_string', 'off_base']]

In [20]:
#Create a final dataframe the joins the time the runner is off the base with the catch timestamp
final = off_base.merge(outfield_catch, how="inner", on="play_string")

In [ ]:
#Creation of the GO TIME (reaction time) of the plays by subtracting the catch_timestamp from off_base
final['Go_Time'] = final['off_base'] - final['catch_timestamp']

In [ ]:
#Joining the final dataframe to include the baserunners on 3rd to be able to keep track of indiviudal players for the graphics
final_with_players = final.merge(interested, how='left', on='play_string')

#Remove the outliers that have a GO TIME above 700 milliseconds
final_with_players = final_with_players[
    final_with_players['Go_Time'] < 700
]